In [3]:
import gzip
import json
import time
import requests
import pandas as pd
from io import BytesIO
from datetime import datetime

# URLs
BASE_URL = "https://transportal.cee.wisc.edu/partners/community-maps/crash/public/crashesKML.do"
SEARCH_PAGE_URL = "https://transportal.cee.wisc.edu/partners/community-maps/crash/search/BasicSearch.do"

START_YEAR = 2000
END_YEAR = datetime.now().year  # 2026
SEVERITIES = ["k", "a"]


def create_authenticated_session():
    """Initialize session and load initial JSESSIONID cookies."""
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:156.0) Gecko/20100101 Firefox/156.0",
        "Accept": "*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Referer": SEARCH_PAGE_URL,
        "X-Requested-With": "XMLHttpRequest",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
    })

    print("Establishing session with WisTransPortal...")
    try:
        # Pre-flight GET to harvest valid JSESSIONID cookies
        response = session.get(SEARCH_PAGE_URL, timeout=30)
        response.raise_for_status()
        print("Session established successfully.")
    except Exception as e:
        print(f"Warning: Failed to load search page landing: {e}")

    return session


def decompress_response(response_content):
    """Attempt gzip decompression or fallback to plain text parsing."""
    try:
        # Decompress gzip payload in memory
        with gzip.GzipFile(fileobj=BytesIO(response_content)) as gz:
            decompressed_bytes = gz.read()
            return json.loads(decompressed_bytes.decode("utf-8"))
    except Exception:
        # Fallback if the server returned plain JSON uncompressed
        return json.loads(response_content.decode("utf-8"))


def fetch_year_data(session, year: int) -> list:
    """Fetch crashes for a given year across severities 'k' and 'a'."""
    year_records = []

    for severity in SEVERITIES:
        params = {
            "compressed": "N",          # Set to Y based on working curl
            "filetype": "json",
            "county": "ALL",
            "startyear": str(year),
            "startmonth": "01",
            "endyear": str(year),
            "endmonth": "12",
            "injsvr": severity,
            "flaglogic": "OR",
            "_": int(time.time() * 1000)  # Timestamp cache-buster
        }

        print(f"Fetching Year {year} | Severity '{severity.upper()}'...")
        try:
            res = session.get(BASE_URL, params=params, timeout=60)
            res.raise_for_status()

            data = decompress_response(res.content)

            # Standardize list or dict structures
            crashes = []
            if isinstance(data, list):
                crashes = data
            elif isinstance(data, dict):
                crashes = data.get("crashes", data.get("features", []))

            print(f"  -> Successfully retrieved {len(crashes)} records.")
            year_records.extend(crashes)

            time.sleep(1)  # Politeness delay

        except Exception as e:
            print(f"  [!] Failed to retrieve {year} ({severity}): {e}")

    return year_records


def scrape_all():
    session = create_authenticated_session()
    all_records = []

    print(f"\nStarting scrape for years {START_YEAR} through {END_YEAR}...")
    for year in range(START_YEAR, END_YEAR + 1):
        data = fetch_year_data(session, year)
        all_records.extend(data)

    if not all_records:
        print("\nNo data collected. Exiting.")
        return

    print(f"\nProcessing {len(all_records)} total raw records...")
    df = pd.DataFrame(all_records)

    # Remove duplicates
    initial_len = len(df)
    dedup_cols = [c for c in ["DOCTNMBR", "id", "CRASHNMBR"] if c in df.columns]
    if dedup_cols:
        df = df.drop_duplicates(subset=dedup_cols[0])
    else:
        df = df.drop_duplicates()

    print(f"Deduplicated {initial_len - len(df)} duplicate records.")

    output_filename = f"wisconsin_crashes_K_A_{START_YEAR}_{END_YEAR}.csv"
    df.to_csv(output_filename, index=False)
    print(f"\nSuccess! Exported {len(df)} records to '{output_filename}'.")


if __name__ == "__main__":
    scrape_all()

Establishing session with WisTransPortal...
Session established successfully.

Starting scrape for years 2000 through 2026...
Fetching Year 2000 | Severity 'K'...
  -> Successfully retrieved 0 records.
Fetching Year 2000 | Severity 'A'...
  -> Successfully retrieved 0 records.
Fetching Year 2001 | Severity 'K'...
  -> Successfully retrieved 691 records.
Fetching Year 2001 | Severity 'A'...
  -> Successfully retrieved 0 records.
Fetching Year 2002 | Severity 'K'...
  -> Successfully retrieved 727 records.
Fetching Year 2002 | Severity 'A'...
  -> Successfully retrieved 0 records.
Fetching Year 2003 | Severity 'K'...
  -> Successfully retrieved 753 records.
Fetching Year 2003 | Severity 'A'...
  -> Successfully retrieved 0 records.
Fetching Year 2004 | Severity 'K'...
  -> Successfully retrieved 714 records.
Fetching Year 2004 | Severity 'A'...
  -> Successfully retrieved 0 records.
Fetching Year 2005 | Severity 'K'...
  -> Successfully retrieved 706 records.
Fetching Year 2005 | Severit

In [7]:
import pandas as pd
import json

In [11]:
pd.read_csv("wisconsin_crashes_K_A_2000_2026.csv")['properties'][61143]

"{'date': '01/01/2026', 'totfatl': '0', 'cnytname': 'RACINE', 'injsvr': 'A', 'muniname': 'RACINE', 'totinj': '5', 'flags': {'teendrvr': 'N', 'distdrvr': 'N', 'occpprot': 'N', 'speedflag': 'Y', 'impaired': 'N', 'conszone': 'N', '65+drvr': 'Y', 'bikeflag': 'N', 'mcycflag': 'N', 'pedflag': 'N'}, 'atrdwy': 'BOYD AVE', 'onrdwy': '16TH ST'}"

In [12]:
pd.read_csv("wisconsin_crashes_K_A_2000_2026.csv")['geometry'][61143]

"{'coordinates': [-87.80813629, 42.712392023], 'type': 'Point'}"

In [13]:
pd.read_csv("wisconsin_crashes_K_A_2000_2026.csv")['properties'][0]

"{'date': '12/31/2001', 'totfatl': '1', 'cnytname': 'ROCK', 'injsvr': 'K', 'muniname': 'BELOIT', 'totinj': '1', 'flags': {'teendrvr': 'N', 'distdrvr': 'Y', 'occpprot': 'Y', 'speedflag': 'Y', 'impaired': 'N', 'conszone': 'N', '65+drvr': 'N', 'bikeflag': 'N', 'mcycflag': 'N', 'pedflag': 'N'}, 'atrdwy': 'COLLEY RD', 'onrdwy': '039'}"

In [14]:
data = pd.read_csv("wisconsin_crashes_K_A_2000_2026.csv")

In [18]:
data['properties'][1]

"{'date': '12/31/2001', 'totfatl': '1', 'cnytname': 'MILWAUKEE', 'injsvr': 'K', 'muniname': 'OAK CREEK', 'totinj': '2', 'flags': {'teendrvr': 'N', 'distdrvr': 'N', 'occpprot': 'N', 'speedflag': 'N', 'impaired': 'N', 'conszone': 'N', '65+drvr': 'Y', 'bikeflag': 'N', 'mcycflag': 'N', 'pedflag': 'N'}, 'atrdwy': 'ZZ', 'onrdwy': '241'}"